In [ ]:
# ============================================================
# CELL 1: DATASET & DATALOADER CHO PRANET TỪ KVASIR_YOLO_SEG_BG20
# ============================================================
import os, glob, cv2, torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

DATASET_ROOT = "/kaggle/input/datasets/okeydokey0805/kvasir-yolo-seg-bg20/Kvasir_YOLO_SEG_BG20"

class KvasirBG20PraNetDataset(Dataset):
    def __init__(self, root_dir, split="train", img_size=(352, 352)):
        self.img_dir = os.path.join(root_dir, "images", split)
        self.label_dir = os.path.join(root_dir, "labels", split)
        self.img_files = sorted(glob.glob(os.path.join(self.img_dir, "*.jpg")))
        self.img_size = img_size
        assert len(self.img_files) > 0, f"Không tìm thấy ảnh tại: {self.img_dir}"
        
    def __len__(self):
        return len(self.img_files)
        
    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        stem = os.path.splitext(os.path.basename(img_path))[0]
        label_path = os.path.join(self.label_dir, stem + ".txt")
        
        # 1. Đọc ảnh RGB
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # 2. Tạo Mask nhị phân on-the-fly từ polygon hoặc ảnh nền
        mask = np.zeros((h, w), dtype=np.uint8)
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) > 5:
                        coords = np.array([float(x) for x in parts[1:]], dtype=np.float32).reshape(-1, 2)
                        coords[:, 0] *= w
                        coords[:, 1] *= h
                        pts = coords.astype(np.int32)
                        cv2.fillPoly(mask, [pts], 255)
                        
        # 3. Resize kích thước chuẩn benchmark 352x352
        img = cv2.resize(img, self.img_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)
        
        # 4. Chuẩn hóa ảnh theo ImageNet chuẩn cho Backbone ResNet
        img = img.astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = (img - mean) / std
        
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        mask_tensor = torch.tensor(mask > 127, dtype=torch.float32).unsqueeze(0) # (1, H, W)
        
        return img_tensor, mask_tensor

# Khởi tạo kiểm tra
train_ds = KvasirBG20PraNetDataset(DATASET_ROOT, split="train", img_size=(352, 352))
val_ds = KvasirBG20PraNetDataset(DATASET_ROOT, split="val", img_size=(352, 352))

print(f"✅ Đã nạp thành công Dataset cho PraNet:")
print(f"   - Tập Train: {len(train_ds)} ảnh (880 polyp + 160 nền)")
print(f"   - Tập Val  : {len(val_ds)} ảnh (120 polyp + 40 nền)")


In [ ]:
# ============================================================
# CELL 2: KHÓA SEED 9, KHỞI TẠO PRANET & HUẤN LUYỆN ĐỐI CHUẨN
# ============================================================
import os, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.models as models

# 1. SIÊU THAM SỐ
SEED = 19
EPOCHS = 100
BATCH_SIZE = 8
LR = 1e-4
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "/kaggle/working/runs_pranet"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. KHÓA TẤT ĐỊNH HỆ THỐNG
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

# 3. CÁC MODULE CƠ BẢN CỦA PRANET
class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride=1, padding=0, dilation=1):
        super().__init__()
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel_size, stride=stride,
                              padding=padding, dilation=dilation, bias=False)
        self.bn = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class RFB(nn.Module):
    def __init__(self, in_channel, out_channel):
        super().__init__()
        self.relu = nn.ReLU(True)
        self.branch0 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1)
        )
        self.branch1 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 3), padding=(0, 1)),
            BasicConv2d(out_channel, out_channel, kernel_size=(3, 1), padding=(1, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=3, dilation=3)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 5), padding=(0, 2)),
            BasicConv2d(out_channel, out_channel, kernel_size=(5, 1), padding=(2, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=5, dilation=5)
        )
        self.branch3 = nn.Sequential(
            BasicConv2d(in_channel, out_channel, 1),
            BasicConv2d(out_channel, out_channel, kernel_size=(1, 7), padding=(0, 3)),
            BasicConv2d(out_channel, out_channel, kernel_size=(7, 1), padding=(3, 0)),
            BasicConv2d(out_channel, out_channel, 3, padding=7, dilation=7)
        )
        self.conv_cat = BasicConv2d(4 * out_channel, out_channel, 3, padding=1)
        self.conv_res = BasicConv2d(in_channel, out_channel, 1)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        x_cat = self.conv_cat(torch.cat((x0, x1, x2, x3), dim=1))
        return self.relu(x_cat + self.conv_res(x))
class Aggregation(nn.Module):
    def __init__(self, channel=32):
        super().__init__()
        self.relu = nn.ReLU(True)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
        self.conv_upsample1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample3 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample4 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv_upsample5 = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)
        
        # Nhận vào x2_2 sau khi upsample 2x lên mức spatial của x3 (44x44)
        self.conv_concat = BasicConv2d(2 * channel, 2 * channel, 3, padding=1)
        self.conv_concat2 = BasicConv2d(3 * channel, 3 * channel, 3, padding=1)
        self.conv5 = nn.Conv2d(3 * channel, 1, 1)

    def forward(self, x1, x2, x3):
        # x1: 11x11, x2: 22x22, x3: 44x44
        x1_1 = x1
        x2_1 = self.conv_upsample1(self.upsample(x1)) * x2                      # 22x22
        x3_1 = self.conv_upsample2(self.upsample(self.upsample(x1))) * self.conv_upsample3(self.upsample(x2)) * x3  # 44x44

        # Ghép x2_1 (22x22) và x1_1 upsampled (22x22) -> x2_2 (22x22)
        x2_2 = torch.cat((x2_1, self.conv_upsample4(self.upsample(x1_1))), 1)
        x2_2 = self.conv_upsample5(x2_2)                                         # (B, 64, 22, 22)

        # Upsample x2_2 từ 22x22 lên 44x44 để khớp với x3_1 (44x44)
        x2_2_up = self.conv_concat(self.upsample(x2_2))                         # (B, 64, 44, 44)

        # Ghép x3_1 (32 channels, 44x44) + x2_2_up (64 channels, 44x44) -> 96 channels, 44x44
        x3_2 = torch.cat((x3_1, x2_2_up), 1)
        x3_2 = self.conv_concat2(x3_2)

        x = self.conv5(x3_2)                                                    # (B, 1, 44, 44)
        return x

class ReverseAttention(nn.Module):
    def __init__(self, channel=32):
        super().__init__()
        self.conv1 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv2 = BasicConv2d(channel, channel, 3, padding=1)
        self.conv3 = nn.Conv2d(channel, 1, 1)

    def forward(self, x, bg):
        # Reverse attention: (1 - Sigmoid(bg)) * x
        x_att = -1 * (torch.sigmoid(bg)) + 1
        x = x * x_att
        x = self.conv1(x)
        x = self.conv2(x)
        return self.conv3(x)

# 4. TOÀN BỘ MẠNG PRANET
class PraNet(nn.Module):
    def __init__(self, channel=32):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.layer0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1  # 256
        self.layer2 = resnet.layer2  # 512
        self.layer3 = resnet.layer3  # 1024
        self.layer4 = resnet.layer4  # 2048

        self.rfb2 = RFB(512, channel)
        self.rfb3 = RFB(1024, channel)
        self.rfb4 = RFB(2048, channel)

        self.agg = Aggregation(channel)
        self.ra4 = ReverseAttention(channel)
        self.ra3 = ReverseAttention(channel)
        self.ra2 = ReverseAttention(channel)

    def forward(self, x):
        h, w = x.shape[2:]
        x0 = self.layer0(x)
        x1 = self.layer1(x0)
        x2 = self.layer2(x1)  # 512
        x3 = self.layer3(x2)  # 1024
        x4 = self.layer4(x3)  # 2048

        x2_rfb = self.rfb2(x2)  # (B, 32, H/8, W/8)
        x3_rfb = self.rfb3(x3)  # (B, 32, H/16, W/16)
        x4_rfb = self.rfb4(x4)  # (B, 32, H/32, W/32)

        # Global Saliency Map từ Aggregation
        sg = self.agg(x4_rfb, x3_rfb, x2_rfb)
        lateral_map_sg = F.interpolate(sg, size=(h, w), mode='bilinear', align_corners=False)

        # Reverse Attention Stage 4
        s4 = self.ra4(x4_rfb, F.interpolate(sg, size=x4_rfb.shape[2:], mode='bilinear', align_corners=False))
        lateral_map_s4 = F.interpolate(s4, size=(h, w), mode='bilinear', align_corners=False)

        # Reverse Attention Stage 3
        s3 = self.ra3(x3_rfb, F.interpolate(s4, size=x3_rfb.shape[2:], mode='bilinear', align_corners=False))
        lateral_map_s3 = F.interpolate(s3, size=(h, w), mode='bilinear', align_corners=False)

        # Reverse Attention Stage 2
        s2 = self.ra2(x2_rfb, F.interpolate(s3, size=x2_rfb.shape[2:], mode='bilinear', align_corners=False))
        lateral_map_s2 = F.interpolate(s2, size=(h, w), mode='bilinear', align_corners=False)

        return lateral_map_s2, lateral_map_s3, lateral_map_s4, lateral_map_sg

# 5. HÀM LOSS (STRUCTURE LOSS: BCE + IOU)
def structure_loss(pred, mask):
    bce = F.binary_cross_entropy_with_logits(pred, mask, reduction='mean')
    pred_prob = torch.sigmoid(pred)
    inter = (pred_prob * mask).sum(dim=(2, 3))
    union = (pred_prob + mask).sum(dim=(2, 3))
    iou = 1.0 - (inter + 1.0) / (union - inter + 1.0)
    return bce + iou.mean()

# 6. DATALOADER & HUẤN LUYỆN
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model = PraNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f"Bắt đầu huấn luyện PraNet trên thiết bị: {DEVICE} với {EPOCHS} epochs...")
best_val_dice = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for imgs, masks in train_loader:
        imgs = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        optimizer.zero_grad()
        out2, out3, out4, out_g = model(imgs)

        # Kết hợp loss từ cả 4 nhánh theo paper PraNet
        loss = structure_loss(out2, masks) + structure_loss(out3, masks) + \
               structure_loss(out4, masks) + structure_loss(out_g, masks)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    avg_loss = total_loss / len(train_loader)

    # Đánh giá nhanh Validation mỗi 5 epochs
    if epoch % 5 == 0 or epoch == EPOCHS:
        model.eval()
        dices = []
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs = imgs.to(DEVICE)
                masks = masks.to(DEVICE)
                pred, _, _, _ = model(imgs)
                pred = (torch.sigmoid(pred) > 0.5).float()
                
                intersection = (pred * masks).sum(dim=(2, 3))
                dice = (2.0 * intersection + 1e-5) / (pred.sum(dim=(2, 3)) + masks.sum(dim=(2, 3)) + 1e-5)
                dices.extend(dice.cpu().numpy().flatten())

        val_dice = np.mean(dices)
        print(f"Epoch [{epoch:03d}/{EPOCHS:03d}] - Train Loss: {avg_loss:.4f} | Val Dice: {val_dice:.4f}")

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "pranet_best.pth"))
            print(f" -> Đã lưu checkpoint tốt nhất tại: {OUTPUT_DIR}/pranet_best.pth")